# Modern Databases 2026, SQL 1: setup and exercises

This notebook creates the Airbnb teaching database for Thursday's session and holds the exercises.
Run it top to bottom once before the laptop phase. It needs DuckDB 1.5.5 and five data files (about 11 MB),
which it downloads from the course repository, [github.com/JanKalo/MDB2026](https://github.com/JanKalo/MDB2026),
unless they are already next to the notebook.

| Table | Rows | Columns |
|---|---|---|
| `listings` | 6,503 | listing_id, host_id, host_name, name, neighbourhood, room_type, property_type, accommodates, price_eur, minimum_nights, rating |
| `reviews` | 443,266 | review_id, listing_id, review_date, reviewer_id, reviewer_name |
| `calendar` | 2,373,598 | listing_id, stay_date, available, minimum_nights, maximum_nights (one row per listing and night, 19 March 2026 to 19 March 2027) |
| `listings_2025` | 10,168 | listing_id (the 2025 snapshot) |
| `airbnb` | 45 | Monday's printed sheet, all 22 columns, with price, minimum_nights and review_scores_rating cast to numbers |

Data: Inside Airbnb, Amsterdam, March 2026 release (listings, reviews and calendar). CC BY 4.0.

In [ ]:
%pip -q install duckdb==1.5.5 pandas

In [ ]:
import duckdb, os, pandas as pd

print("DuckDB", duckdb.__version__)
assert duckdb.__version__.startswith("1.5"), "Install DuckDB 1.5.5: pip install duckdb==1.5.5 (do not upgrade during the course)"
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)

## 1. Where the files are

The cell below looks for the five files next to this notebook or in a folder called `data`. Whatever it does not
find it reads from the course repository on GitHub, so with an internet connection nothing has to be downloaded by hand.

In [ ]:
FILES = ["listings.parquet", "reviews.parquet", "calendar.parquet", "listings_2025.parquet",
         "Lecture1_Airbnb_Handout_45x22.csv"]

BASE = "https://raw.githubusercontent.com/JanKalo/MDB2026/main/data/"   # the course repository

def locate(name):
    for folder in [".", "data", "../data/parquet", "../jupyter", "../data"]:
        candidate = os.path.join(folder, name)
        if os.path.exists(candidate):
            return candidate
    return BASE + name

paths = {name: locate(name) for name in FILES}
paths

## 2. Create the database

The tables are written to the file `mdb2026-sql1.duckdb`, so the next time you open this notebook the data is
already there. Delete that file if you want to start again.

In [ ]:
con = duckdb.connect("mdb2026-sql1.duckdb")

for table in ["listings", "reviews", "calendar", "listings_2025"]:
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_parquet(?)", [paths[table + ".parquet"]])

# Monday's printed sheet: the raw columns, with the three numeric columns cast once.
con.execute("""
CREATE OR REPLACE TABLE airbnb AS
SELECT * REPLACE (
    TRY_CAST(REPLACE(REGEXP_REPLACE(price, '[^0-9.,]', ''), ',', '') AS DOUBLE) AS price,
    TRY_CAST(minimum_nights AS INTEGER) AS minimum_nights,
    TRY_CAST(review_scores_rating AS DOUBLE) AS review_scores_rating)
FROM read_csv_auto(?, header=True, nullstr=['', 'NA', 'NaN'])
""", [paths["Lecture1_Airbnb_Handout_45x22.csv"]])

expected = {"listings": 6503, "reviews": 443266, "calendar": 2373598, "listings_2025": 10168, "airbnb": 45}
for table, n in expected.items():
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:14s} {count:>10,}  {'ok' if count == n else 'expected ' + format(n, ',')}")

## 3. Running a query

`q()` runs one SQL statement and shows the result as a table. Every exercise below has an empty cell; write the
query inside the triple quotes. A checkpoint gives the number you should get, not the query.

In [ ]:
def q(sql):
    result = con.sql(sql)
    if result is None:
        print("This cell has no query yet.")
        return None
    return result.df()

q("SELECT * FROM listings LIMIT 5")

In [ ]:
q("DESCRIBE calendar")

## 4. Step 1: Monday's five questions on all 6,503 listings

On Monday you answered these on 45 printed rows. Now answer them on the whole `listings` table.
Column names differ from the sheet: `neighbourhood` instead of `neighbourhood_cleansed`, `price_eur` instead of `price`.

| | Question | Checkpoint |
|---|---|---|
| Q1 | How many distinct hosts are there? | 5,474 by `host_id`; 2,655 by `host_name`. Two hosts can share a name, so count `host_id`. |
| Q2 | Which neighbourhood has the highest mean price? | Centrum-Oost, 466.77 |
| Q3 | Which neighbourhood has the highest share of missing prices? | Westerpark, 19.3% |
| Q4 | Median `minimum_nights` for entire homes? | 2 |
| Q5 | Cheapest neighbourhood on average? | Bijlmer-Centrum, 160.75. Still the cheapest (148.06) if a missing price counts as zero. |

In [ ]:
q("""
-- Q1: distinct hosts, by host_id and by host_name

""")

In [ ]:
q("""
-- Q2: mean price per neighbourhood, highest first

""")

In [ ]:
q("""
-- Q3: share of listings without a price, per neighbourhood

""")

In [ ]:
q("""
-- Q4: median minimum_nights for room_type = 'Entire home/apt'

""")

In [ ]:
q("""
-- Q5: cheapest neighbourhood on average, and the same with COALESCE(price_eur, 0)

""")

## 5. Your topic

Do tasks 1 and 2 of your topic, then choose task 3 or 4. Before writing, say what one row of the result represents.

### Topic A: availability

The calendar holds one row per listing and night. Arrival is inclusive and checkout is exclusive: arrival on
3 September and checkout on 6 September means the nights of 3, 4 and 5 September.
A percentage is an average of ones and zeros: `AVG(CASE WHEN available THEN 1 ELSE 0 END)`.

| | Task | Checkpoint |
|---|---|---|
| A1 | Count the available listing-nights for arrival 3 September and checkout 6 September 2026. | 8,607 |
| A2 | Find the listings marked available on every night of that stay. | 2,712 rows |
| A3 | Of those, which also permit a three-night stay? (`minimum_nights` at most 3 and `maximum_nights` at least 3 on every night.) | 2,021 rows |
| A4 | Availability percentage per neighbourhood over the whole calendar. | from 50.6% (Bijlmer-Centrum) to 27.6% (Noord-West) |

In [ ]:
q("""
-- A1

""")

In [ ]:
q("""
-- A2

""")

In [ ]:
q("""
-- A3 or A4

""")

### Topic B: neighbourhoods

`price_eur` is NULL for 916 listings. `AVG` and `MEDIAN` ignore them; decide whether that is what the question means.

| | Task | Checkpoint |
|---|---|---|
| B1 | Per neighbourhood: number of listings, number with a known price, mean and median price. | 22 rows; the highest median is De Pijp - Rivierenbuurt, 342.59 |
| B2 | Keep the neighbourhoods with at least 200 known prices and a mean below 350. | 4 rows: Bos en Lommer, Oostelijk Havengebied - Indische Buurt, Oud-Oost, Westerpark |
| B3 | Neighbourhoods whose mean price exceeds the citywide mean. | 6 rows; the citywide mean is 359.25 |
| B4 | Listings priced above their own neighbourhood's mean, using a CTE. | 2,030 rows |

In [ ]:
q("""
-- B1

""")

In [ ]:
q("""
-- B2

""")

In [ ]:
q("""
-- B3 or B4

""")

### Topic C: hosts

| | Task | Checkpoint |
|---|---|---|
| C1 | Hosts with listings in at least three neighbourhoods. | 9 rows |
| C2 | Hosts with a listing in Centrum-West and one in Centrum-Oost. | 17 rows |
| C3 | Self-join `listings` to pair two listings of the same host in different neighbourhoods. | 1,206 rows if each pair appears twice |
| C4 | Return each pair once, with `a.listing_id < b.listing_id`. | 603 rows over 55 hosts |

In [ ]:
q("""
-- C1

""")

In [ ]:
q("""
-- C2

""")

In [ ]:
q("""
-- C3 or C4

""")

### Topic D: reviews

Dates compare with `DATE '2025-01-01'`. Every `listing_id` in `reviews` also occurs in `listings`.

| | Task | Checkpoint |
|---|---|---|
| D1 | Per neighbourhood: number of reviews written in 2025 and number of distinct listings reviewed. | 22 rows; 82,031 reviews in total; Centrum-West has 17,951 |
| D2 | Current listings that have never received a review. | 719 rows |
| D3 | The five listings with the most reviews in 2025. | 959, 720, 684, 641 and 440 reviews |
| D4 | The listing with the most reviews in each neighbourhood, ties included. | 22 rows (there are no ties) |

In [ ]:
q("""
-- D1

""")

In [ ]:
q("""
-- D2

""")

In [ ]:
q("""
-- D3 or D4

""")

### If you finish early: five harder tasks

These go beyond the topics. Solutions are on slides 63 and 64.

| | Task | Checkpoint |
|---|---|---|
| H1 | Hosts with at least three listings, all of which are entire homes. | 47 hosts (184 hosts have three or more listings) |
| H2 | Neighbourhoods where the average number of reviews per listing is above the average over all listings. Listings without a review count as zero. | 6 rows, Osdorp first with 295.2; the figure over all listings is 68.2 |
| H3 | Listings that received a review in 2025 and are free on all three nights of the stay (arrival 3 September, checkout 6 September). | 2,040 rows |
| H4 | Listings that are free on every Saturday in September 2026. Do not count the Saturdays by hand; `dayofweek(stay_date) = 6` is a Saturday. | 1,930 rows; there are 4 Saturdays |
| H5 | The query below is meant to return the listings without a review, but it returns 0 rows. Say why, then fix it. | 719 rows once fixed |

```sql
SELECT l.listing_id
FROM listings AS l
JOIN reviews AS r ON r.listing_id = l.listing_id
WHERE r.review_id IS NULL;
```

In [ ]:
q("""
-- H1

""")

In [ ]:
q("""
-- H2

""")

In [ ]:
q("""
-- H3

""")

In [ ]:
q("""
-- H4

""")

In [ ]:
q("""
-- H5, fixed

""")

## 6. The paper exercises, checked in the database

The sheet is the table `airbnb`, with the column names as printed (`price`, `minimum_nights`,
`review_scores_rating`, `neighbourhood_cleansed`). Paper 1 returns 2 rows. Paper 2 returns 35 rows, and 45 once
missing prices are kept. Paper 3 can be run on the six listings of the excerpt: with `listing_id IN (8191077,
44374975, 4488692, 27711346, 4210531, 97221)` in the WHERE clause, the query should return A, C and F, that is
8191077, 4488692 and 97221.

In [ ]:
q("""
-- paper 1

""")

In [ ]:
q("""
-- paper 2

""")

In [ ]:
q("""
-- paper 3, on the six listings of the excerpt

""")

## 7. When a result looks wrong

1. What does one row represent: a listing, a review, a night or a group?
2. Did a join multiply rows, or did unmatched rows disappear?
3. Did a comparison with NULL evaluate to UNKNOWN, or did an aggregate ignore NULL values?
4. Is the condition about rows (WHERE) or about groups (HAVING)?
5. If several rows tie for the maximum, are all of them returned?

Go back to the logical order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY rather than editing syntax at random.